# Cluster 2 Stacking Model - Alternative Implementation

## Objective
Build an independent stacking model for Cluster 2 using a different architectural approach to provide model diversity.

## Key Features
- **Base Models**: XGBoost, DecisionTree with class imbalance handling
- **Meta Model**: GradientBoosting with limited complexity to prevent overfitting
- **Preprocessing**: RobustScaler for outlier resilience
- **CV Strategy**: RepeatedStratifiedKFold for robust validation
- **Advanced Features**: Passthrough enabled, optimized for extreme imbalance

## Data Context
- Cluster 2: 2059 rows, 6 bankruptcies (0.29% - extreme class imbalance)

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import recall_score, make_scorer, confusion_matrix, classification_report
from sklearn.metrics import average_precision_score, f1_score, precision_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, StackingClassifier

# Import gradient boosting libraries
try:
    from lightgbm import LGBMClassifier
    lgbm_available = True
except ImportError:
    lgbm_available = False

try:
    from xgboost import XGBClassifier
    xgb_available = True
except ImportError:
    xgb_available = False

RANDOM_STATE = 42
print("Libraries loaded successfully.")

In [ ]:
# Load cluster data
df_cluster2 = pd.read_csv("cluster_2.csv")
print(f"Data loaded. Shape: {df_cluster2.shape}")

# Load feature names
top40 = joblib.load("top_features_for_clustering.joblib")
features_to_use = top40

X_sub = df_cluster2[features_to_use].copy()
y_sub = df_cluster2["Bankrupt?"].copy()

print(f"\nFeatures: {X_sub.shape}")
print(f"Target Distribution:\n{y_sub.value_counts()}")
print(f"\nPositive class ratio: {y_sub.sum() / len(y_sub):.4%}")
print(f"Class imbalance ratio: {(y_sub == 0).sum() / (y_sub == 1).sum():.1f}:1")

## Base Model Configuration

Using diverse base models with appropriate imbalance handling

In [ ]:
# Calculate scale_pos_weight for imbalance handling
scale_pos_weight = (y_sub == 0).sum() / (y_sub == 1).sum()
print(f"Scale pos weight for imbalance: {scale_pos_weight:.2f}")

base_estimators = []

# Base Model 1: LightGBM (if available)
if lgbm_available:
    lgbm = LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=5,
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        verbose=-1,
        force_col_wise=True
    )
    base_estimators.append(('lgbm', lgbm))
    print("✓ Added LightGBM")

# Base Model 2: XGBoost (if available)
if xgb_available:
    xgb = XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=5,
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        eval_metric='logloss',
        verbosity=0
    )
    base_estimators.append(('xgb', xgb))
    print("✓ Added XGBoost")

# Base Model 3: DecisionTree with tuning
dt = DecisionTreeClassifier(
    max_depth=8,
    min_samples_split=50,
    min_samples_leaf=20,
    class_weight='balanced',
    random_state=RANDOM_STATE
)
base_estimators.append(('dt', dt))
print("✓ Added DecisionTree")

# Fallback if neither LGBM nor XGB available
if not lgbm_available and not xgb_available:
    gb_base = GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.1,
        max_depth=4,
        random_state=RANDOM_STATE
    )
    base_estimators.append(('gb_base', gb_base))
    print("✓ Added GradientBoosting")

print(f"\nTotal base models: {len(base_estimators)}")

## Meta Model Configuration

Simple GradientBoosting classifier to avoid overfitting

In [ ]:
# Meta model with limited complexity
meta_model = GradientBoostingClassifier(
    n_estimators=15,
    max_depth=2,
    learning_rate=0.1,
    random_state=RANDOM_STATE
)

print("Meta model: GradientBoostingClassifier")
print(f"  - n_estimators: 15")
print(f"  - max_depth: 2")

## Build Stacking Pipeline

Using passthrough=True to provide meta-model access to original features

In [ ]:
# CV Strategy for robust validation
cv_strategy = RepeatedStratifiedKFold(
    n_splits=3,
    n_repeats=10,
    random_state=RANDOM_STATE
)

# Stacking Classifier
stacking_clf = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta_model,
    cv=cv_strategy,
    stack_method='predict_proba',
    passthrough=True,
    n_jobs=-1
)

# Complete pipeline with preprocessing
model_pipeline = Pipeline([
    ('scaler', RobustScaler()),
    ('stacking', stacking_clf)
])

print("\n" + "="*60)
print("STACKING PIPELINE ARCHITECTURE")
print("="*60)
print(f"Preprocessing: RobustScaler")
print(f"Base Models: {[name for name, _ in base_estimators]}")
print(f"Meta Model: GradientBoostingClassifier (n_est=15, depth=2)")
print(f"CV Strategy: RepeatedStratifiedKFold(3 splits, 10 repeats)")
print(f"Passthrough: True")
print(f"Features: All {len(features_to_use)} features")
print("="*60)

## Cross-Validation Evaluation

In [ ]:
print("\nPerforming cross-validation...")
print("This may take a few minutes...\n")

recall_scorer = make_scorer(recall_score, pos_label=1, zero_division=0)

cv_scores = cross_val_score(
    model_pipeline,
    X_sub,
    y_sub,
    cv=cv_strategy,
    scoring=recall_scorer,
    n_jobs=-1
)

print("Cross-Validation Results (Recall for Bankrupt class):")
print(f"  Mean: {cv_scores.mean():.4f}")
print(f"  Std:  {cv_scores.std():.4f}")
print(f"  Min:  {cv_scores.min():.4f}")
print(f"  Max:  {cv_scores.max():.4f}")

## Train Final Model

In [ ]:
print("\nFitting final model...")
model_pipeline.fit(X_sub, y_sub)
print("✓ Model fitted successfully")

# Predictions
y_pred = model_pipeline.predict(X_sub)
y_pred_proba = model_pipeline.predict_proba(X_sub)[:, 1]

# Confusion Matrix
cm = confusion_matrix(y_sub, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

# Metrics for Table 3
TT = int(tp)
TF = int(fn)
N_features = len(features_to_use)
eq1_acc = TT / (TF + TT) if (TF + TT) > 0 else 0.0

print("\n" + "="*60)
print("RESULTS FOR TABLE 3")
print("="*60)
print(f"Confusion Matrix:")
print(f"  [[TN={tn:4d}, FP={fp:4d}]")
print(f"   [FN={fn:4d}, TP={tp:4d}]]")
print(f"\nTable 3 Metrics:")
print(f"  TT (True Bankrupts Caught): {TT}")
print(f"  TF (Bankrupts Missed):      {TF}")
print(f"  N_features:                 {N_features}")
print(f"  Eq(1) Accuracy (Recall):    {eq1_acc:.4f}")
print("="*60)

## Additional Performance Metrics

In [ ]:
precision = precision_score(y_sub, y_pred, zero_division=0)
f1 = f1_score(y_sub, y_pred, zero_division=0)
avg_precision = average_precision_score(y_sub, y_pred_proba)

print("\nAdditional Performance Metrics:")
print(f"  Precision:          {precision:.4f}")
print(f"  F1-Score:           {f1:.4f}")
print(f"  Average Precision:  {avg_precision:.4f}")

print("\nClassification Report:")
print(classification_report(y_sub, y_pred, target_names=['Non-Bankrupt', 'Bankrupt'], zero_division=0))

## Save Model Package

In [ ]:
# Package model components
cluster2_package = {
    "cluster_id": 2,
    "features": features_to_use,
    "pipeline": model_pipeline,
    "table3_stats": {
        "TT": TT, 
        "TF": TF, 
        "Eq1_acc": eq1_acc, 
        "N_features": N_features
    },
    "architecture": {
        "base_models": [name for name, _ in base_estimators],
        "meta_model": "GradientBoostingClassifier",
        "scaler": "RobustScaler",
        "cv": "RepeatedStratifiedKFold(3, 10)",
        "passthrough": True
    }
}

# Save
joblib.dump(cluster2_package, "cluster2_stacking_D.joblib")
print("\n" + "="*60)
print("✓ Successfully saved: cluster2_stacking_D.joblib")
print("="*60)